<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">INFO</h1>

In this script, we run player `Dijkstra` in a maze, to visualize its behavior.

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">IMPORTS</h1>

In [29]:
# External imports
import sys
import os
import pprint

# Add needed directories to the path
sys.path.append(os.path.join("..", "players"))

# PyRat imports
from pyrat import Game, GameMode, StartingLocation
from GreedyEachTurn import GreedyEachTurn
from GreedyEachCheese import GreedyEachCheese
from Greedy import Greedy

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">CONSTANTS</h1>

Let's configure the game with a dictionary.

In [9]:
# Customize the game elements
CONFIG = {"mud_percentage": 20.0,
          "nb_cheese": 20,

          "trace_length": 1000,
          "game_mode": GameMode.SEQUENTIAL}

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">RUN THE GAME</h1>

We perform a single game with the configuration defined above.

In [10]:
# Instantiate a game with specified arguments
game = Game(**CONFIG)

# Instantiate a player and add it to the game
player_greedy = Greedy()
player_greedy_each_turn = GreedyEachTurn()

game.add_player(player_greedy, team="Team Ratz", location=StartingLocation.RANDOM)
game.add_player(player_greedy_each_turn, team="Team Pythonz", location=StartingLocation.RANDOM)

# Start the game and
stats = game.start()

We visualize results using a pretty printer.

In [11]:
# Show statistics
pprint.pprint(stats)

{'players': {'Greedy': {'actions': {'east': 12,
                                    'error': 0,
                                    'miss': 0,
                                    'mud': 50,
                                    'north': 15,
                                    'nothing': 0,
                                    'south': 12,
                                    'wall': 0,
                                    'west': 18},
                        'preprocessing_duration': 0.015625,
                        'score': 3.0,
                        'turn_durations': [0.0,
                                           0.0,
                                           0.0,
                                           0.0,
                                           0.0,
                                           0.0,
                                           0.0,
                                           0.0,
                                           0.0,
                                    

### Statistic comparison

In [61]:
import tqdm.auto as tqdm
import numpy as np
import scipy

NB_GAMES = 100

# Customize the game elements
CONFIG = {"mud_percentage": 20.0,
          "game_mode": GameMode.SIMULATION}

# List of winner for each game, 1 if GreedyEachTurn wins
winner = []

for seed in tqdm.tqdm(range(NB_GAMES)):
    # Make the game with given seed
    game = Game(random_seed=seed, nb_cheese=20, **CONFIG)

    player_greedy = Greedy()
    player_greedy_each_turn = GreedyEachTurn()

    game.add_player(player_greedy, team="Team Ratz", location=StartingLocation.RANDOM)
    game.add_player(player_greedy_each_turn, team="Team Pythonz", location=StartingLocation.RANDOM)

    stats = game.start()
    # Store the number of turns needed
    if stats["players"]["GreedyEachTurn"]['score'] > stats["players"]["Greedy"]['score']:
        winner.append(1)
    else:
        winner.append(0)

100%|██████████| 100/100 [00:14<00:00,  6.73it/s]


In [62]:
print("Nombre de parties gagnés par GreedyEachTurn : ", end='')
print(np.count_nonzero(winner))

t_stat, p_value = scipy.stats.ttest_1samp(winner, popmean=0.5)

print("One sample T-test p-value : ", end='')
print(p_value)

Nombre de parties gagnés par GreedyEachTurn : 72
One sample T-test p-value : 4.1441780894966464e-06


On constate donc que GreedyEachTurn gagne plus souvent que Greedy à 72 parties contre 28.\
La valeur p-value étant très inférieur à 1, on en conclue que GreedyEachCheese n'est pas meilleur que Greedy que par hasard.

#### Même chose entre GreedyEachTurn et GreedyEachCheese

In [63]:
# List of winner for each game, 1 if GreedyEachTurn wins
winner_2 = []

for seed in tqdm.tqdm(range(NB_GAMES)):
    # Make the game with given seed
    game = Game(random_seed=seed, nb_cheese=20, **CONFIG)

    player_greedy_each_cheese = GreedyEachCheese()
    player_greedy_each_turn = GreedyEachTurn()

    game.add_player(player_greedy_each_cheese, team="Team Ratz", location=StartingLocation.RANDOM)
    game.add_player(player_greedy_each_turn, team="Team Pythonz", location=StartingLocation.RANDOM)

    stats = game.start()
    # Store the number of turns needed
    if stats["players"]["GreedyEachTurn"]['score'] > stats["players"]["GreedyEachCheese"]['score']:
        winner_2.append(1)
    else:
        winner_2.append(0)

100%|██████████| 100/100 [00:16<00:00,  6.23it/s]


In [64]:
print("Nombre de parties gagnés par GreedyEachTurn : ", end='')
print(np.count_nonzero(winner_2))

t_stat, p_value = scipy.stats.ttest_1samp(winner_2, popmean=0.5)

print("One sample T-test p-value : ", end='')
print(p_value)

Nombre de parties gagnés par GreedyEachTurn : 52
One sample T-test p-value : 0.6912566363051549


On remarque cette fois si que l'écart n'est plus du tout significatif, bien que statistiquement GreedyEachTurn gagne plus souvent, il est aussi plus complexe vu les appels répetés à update_metagraph(). Pour gagner il est donc préférable de garder la vérification à chaque tour.